In [1]:
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.manifold import TSNE
from sklearn.metrics import  accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import TensorDataset, DataLoader
from xgboost import XGBClassifier

In [2]:
CLEAN_CSV       = "clean_sample_150000.csv"
ADV_CSV         = "adv_sample_150000.csv"
N_SAMPLES       = 5000        # adversarial samples to craft per step size
STEP_SIZES      = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
MAX_STEPS       = 50          # coordinate descent steps per sample
RANDOM_STATE    = 42

In [3]:
def load_and_encode(path, le=None, fit=False):
    df = pd.read_csv(path)
    if fit:
        le = LabelEncoder()
        df["label"] = le.fit_transform(df["label"])
    else:
        df["label"] = le.transform(df["label"])
    X = df.drop(columns=["label"], errors="ignore").select_dtypes(include=[np.number])
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X.fillna(0, inplace=True)
    y = df["label"].to_numpy()
    return X, y, le

In [4]:
def train_baseline(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    X_train = X_train.astype("float32")
    X_test  = X_test.astype("float32")
    model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.03, eval_metric="mlogloss", random_state=RANDOM_STATE, tree_method="hist")
    model.fit(X_train, y_train, verbose=False)
    return model, X_test, y_test

In [5]:
def zeroth_order_boundary(model, row_values, feature_cols, true_label, step_size, max_steps, rng):
    X_curr = row_values.copy()
    feat_order = rng.permutation(len(feature_cols))
    def confidence(x):
        df_x = pd.DataFrame([x], columns=feature_cols)
        return model.predict_proba(df_x)[0][true_label]
    def prediction(x):
        df_x = pd.DataFrame([x], columns=feature_cols)
        return model.predict(df_x)[0]
    for _ in range(max_steps):
        for fi in feat_order:
            for direction in [+1, -1]:
                X_try = X_curr.copy()
                X_try[fi] += direction * step_size
                if prediction(X_try) != true_label:
                    return X_try, true_label
                if confidence(X_try) < confidence(X_curr):
                    X_curr = X_try
                    break
    return None, None

In [6]:
def generate_blackbox_adversarials(model, ae_X, ae_y, feature_cols, n_samples, step_size, random_state):
    rng = np.random.default_rng(random_state)
    preds = model.predict(ae_X.astype("float32"))
    correct_mask = (preds == ae_y)
    ae_X_c = ae_X[correct_mask].reset_index(drop=True)
    ae_y_c = ae_y[correct_mask]
    total_eligible = correct_mask.sum()
    print(f"  Eligible (correctly classified): {total_eligible}")
    crafted_rows, true_labels, pred_labels = [], [], []
    attempts = min(n_samples, len(ae_X_c))
    for i in range(attempts):
        row_vals = ae_X_c.iloc[i].values.astype("float32")
        true_label = ae_y_c[i]
        X_adv, label = zeroth_order_boundary(model, row_vals, feature_cols, true_label, step_size=step_size, max_steps=MAX_STEPS, rng=rng)
        if X_adv is None: continue
        pred_after = model.predict(pd.DataFrame([X_adv], columns=feature_cols))[0]
        crafted_rows.append(X_adv)
        true_labels.append(label)
        pred_labels.append(pred_after)
        if (i + 1) % 500 == 0: print(f"  Processed {i+1}/{attempts} — crafted so far: {len(crafted_rows)}")
    print(f"  Crafted: {len(crafted_rows)} / {attempts}")
    result_df = pd.DataFrame(crafted_rows, columns=feature_cols)
    result_df["true_label"] = true_labels
    result_df["y_pred"]     = pred_labels
    return result_df

In [ ]:
X_clean, y_clean, le = load_and_encode(CLEAN_CSV, fit=True)
X_adv, y_adv, _ = load_and_encode(ADV_CSV, le=le, fit=False)
feature_cols = list(X_clean.columns)
model, X_test, y_test = train_baseline(X_clean, y_clean)
summary_rows = []
for step_size in STEP_SIZES:
    print(f"── step_size={step_size} ──────────────────────────────")
    ae_df = generate_blackbox_adversarials(model, X_adv, y_adv, feature_cols, n_samples=N_SAMPLES, step_size=step_size, random_state=RANDOM_STATE)
    out_path = f"blackbox_poison_step{step_size}.csv"
    ae_df.to_csv(out_path, index=False)
    print(f"  Saved → {out_path}  ({len(ae_df)} rows)\n")
    summary_rows.append({
            "step_size":   step_size,
            "n_crafted":   len(ae_df),
            "output_file": out_path
    })
summary = pd.DataFrame(summary_rows)
print("\n══ Summary ══")
print(summary.to_string(index=False))

── step_size=0.01 ──────────────────────────────
  Eligible (correctly classified): 136249
